# IDEA-004: NUPL Exit Trigger

**Hypothesis:** NUPL > 0.75 signals euphoria = time to exit.

**Logic:**
- NUPL = Net Unrealized Profit/Loss = (Market Cap - Realized Cap) / Market Cap
- NUPL > 0.75 means 75%+ of market cap is unrealized profit
- Historically marks cycle tops (extreme greed)
- Alternative to MVRV for exit trigger

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("NUPL Exit Trigger Exploration 🔍")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

nupl = pd.read_parquet(DATA_DIR / "nupl.parquet").rename(columns={"value": "nupl"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")

df = nupl.join(price, how='inner').join(mvrv, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner')
df = df.sort_index()

print(f"Data: {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")

---
## 1. Understand the Data

In [ ]:
# Basic stats
print("NUPL STATISTICS")
print("="*50)
print(f"Min: {df['nupl'].min():.3f}")
print(f"Max: {df['nupl'].max():.3f}")
print(f"Mean: {df['nupl'].mean():.3f}")
print(f"Median: {df['nupl'].median():.3f}")
print(f"Current: {df['nupl'].iloc[-1]:.3f}")

print(f"\nPercentiles:")
for p in [5, 10, 25, 50, 75, 90, 95]:
    print(f"  {p}th: {df['nupl'].quantile(p/100):.3f}")

In [ ]:
# How often is NUPL above various thresholds?
print("\nFREQUENCY ABOVE THRESHOLDS")
print("="*50)
for thresh in [0.5, 0.6, 0.65, 0.7, 0.75, 0.8]:
    days_above = (df['nupl'] > thresh).sum()
    pct = days_above / len(df) * 100
    print(f"NUPL > {thresh}: {days_above} days ({pct:.1f}%)")

In [ ]:
# Visualize NUPL with price
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4],
                    subplot_titles=['BTC Price', 'NUPL'])

fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price'), row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['nupl'], name='NUPL',
                         line=dict(color='green')), row=2, col=1)

# Add threshold lines
fig.add_hline(y=0.75, line_dash='dash', line_color='red', row=2, col=1, annotation_text='Euphoria 0.75')
fig.add_hline(y=0.5, line_dash='dot', line_color='orange', row=2, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_layout(height=600, title_text='NUPL - Exit Zone When > 0.75?')
fig.show()

In [ ]:
# When was NUPL > 0.75 (euphoria)?
high_nupl = df[df['nupl'] > 0.75].copy()
print(f"\nPERIODS WITH NUPL > 0.75 (EUPHORIA)")
print("="*60)
print(f"Total days: {len(high_nupl)}")

if len(high_nupl) > 0:
    high_nupl['gap'] = (high_nupl.index.to_series().diff() > pd.Timedelta(days=30)).cumsum()
    periods = high_nupl.groupby('gap').agg({
        'nupl': ['max', 'mean'],
        'price': ['first', 'max', 'last']
    })
    periods.columns = ['max_nupl', 'avg_nupl', 'start_price', 'max_price', 'end_price']
    
    period_dates = high_nupl.groupby('gap').apply(lambda x: (x.index.min(), x.index.max()))
    
    print(f"\nDistinct euphoria periods: {len(periods)}")
    print("\n" + "-"*80)
    for i, (idx, row) in enumerate(periods.iterrows()):
        start, end = period_dates.iloc[i]
        duration = (end - start).days + 1
        print(f"{start.date()} to {end.date()} ({duration} days)")
        print(f"  NUPL: max={row['max_nupl']:.3f}, avg={row['avg_nupl']:.3f}")
        print(f"  Price: ${row['start_price']:,.0f} → max ${row['max_price']:,.0f} → ${row['end_price']:,.0f}")
        print()

---
## 2. Compare NUPL to MVRV

In [ ]:
# Correlation
print("CORRELATION: NUPL vs MVRV")
print("="*50)
print(f"Correlation: {df['nupl'].corr(df['mvrv']):.3f}")

# When MVRV > 2.25, what's NUPL?
mvrv_high = df['mvrv'] > 2.25
print(f"\nWhen MVRV > 2.25:")
print(f"  Avg NUPL: {df.loc[mvrv_high, 'nupl'].mean():.3f}")
print(f"  Min NUPL: {df.loc[mvrv_high, 'nupl'].min():.3f}")
print(f"  Max NUPL: {df.loc[mvrv_high, 'nupl'].max():.3f}")

# When NUPL > 0.75, what's MVRV?
nupl_high = df['nupl'] > 0.75
if nupl_high.sum() > 0:
    print(f"\nWhen NUPL > 0.75:")
    print(f"  Avg MVRV: {df.loc[nupl_high, 'mvrv'].mean():.2f}")
    print(f"  Min MVRV: {df.loc[nupl_high, 'mvrv'].min():.2f}")
    print(f"  Max MVRV: {df.loc[nupl_high, 'mvrv'].max():.2f}")

In [ ]:
# Scatter plot NUPL vs MVRV
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df['mvrv'],
    y=df['nupl'],
    mode='markers',
    marker=dict(size=3, opacity=0.5),
    name='Data'
))

fig.add_hline(y=0.75, line_dash='dash', line_color='red')
fig.add_vline(x=2.25, line_dash='dash', line_color='green')

fig.update_layout(
    title='NUPL vs MVRV - They measure similar things!',
    xaxis_title='MVRV',
    yaxis_title='NUPL',
    height=500
)
fig.show()

---
## 3. Test NUPL as Exit Trigger

In [ ]:
# Backtest with NUPL exit instead of MVRV
df_test = df[df.index >= '2018-12-15'].copy()
close = df_test['price']

# Create SOPR entry signal
sopr_signal = (df_test['sopr'] < 1) & (df_test['sopr_sth'] < 1)
entries = sopr_signal & ~sopr_signal.shift(1).fillna(False)

In [ ]:
def backtest_nupl_exit(
    df, entries,
    nupl_trigger=0.75,
    trailing_pct=0.20,
    stop_loss=0.20,
    max_hold_days=365
):
    """Use NUPL to trigger trailing stop exit."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        trailing_active = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_nupl = df['nupl'].iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            
            # NUPL triggers trailing stop
            if not trailing_active and current_nupl >= nupl_trigger:
                trailing_active = True
            
            if trailing_active:
                trail_stop = peak_price * (1 - trailing_pct)
                if current_price <= trail_stop:
                    exit_date = current_date
                    exit_price = trail_stop
                    exit_reason = 'nupl_trail'
                    break
            
            if not trailing_active and stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'exit_date': exit_date,
            'entry_price': entry_price,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
# Test different NUPL thresholds
thresholds = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]

print("NUPL EXIT THRESHOLD COMPARISON (In-Sample)")
print("="*100)
print(f"{'Threshold':<12} {'Trades':>10} {'Return':>12} {'Win Rate':>10} {'NUPL Exits':>12} {'Avg Days':>10}")
print("-"*100)

nupl_results = []

for thresh in thresholds:
    trades = backtest_nupl_exit(df_test, entries, nupl_trigger=thresh)
    
    total_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
    win_rate = (trades['pnl_pct'] > 0).mean() if len(trades) > 0 else 0
    nupl_exits = (trades['exit_reason'] == 'nupl_trail').sum()
    avg_days = trades['days_held'].mean() if len(trades) > 0 else 0
    
    print(f"NUPL > {thresh:<5} {len(trades):>10} {total_return*100:>11.0f}% "
          f"{win_rate*100:>9.0f}% {nupl_exits:>12} {avg_days:>10.0f}")
    
    nupl_results.append({
        'threshold': thresh,
        'trades': len(trades),
        'total_return': total_return,
        'win_rate': win_rate,
        'nupl_exits': nupl_exits,
        'trades_df': trades
    })

---
## 4. Walk-Forward Validation

In [ ]:
def walk_forward_nupl(df, nupl_trigger, trailing_pct=0.20):
    """Walk-forward validation with NUPL exit."""
    results = []
    close = df['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        # SOPR entry
        sopr_sig = (test_df['sopr'] < 1) & (test_df['sopr_sth'] < 1)
        test_entries = sopr_sig & ~sopr_sig.shift(1).fillna(False)
        
        trades = backtest_nupl_exit(test_df, test_entries, nupl_trigger, trailing_pct)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'strat_return': strat_return,
            'hold_return': hold_return,
            'beat_hold': strat_return > hold_return,
            'n_trades': len(trades)
        })
    
    wf_df = pd.DataFrame(results)
    return wf_df['beat_hold'].mean(), (wf_df['strat_return'] - wf_df['hold_return']).mean(), wf_df['n_trades'].sum()

In [ ]:
# Walk-forward for each threshold
print("\nWALK-FORWARD VALIDATION")
print("="*80)
print(f"{'Threshold':<12} {'Beat Rate':>15} {'Avg Excess':>15} {'Total Trades':>15}")
print("-"*80)

wf_results = []

for thresh in thresholds:
    beat_rate, avg_excess, total_trades = walk_forward_nupl(df_test, thresh)
    
    print(f"NUPL > {thresh:<5} {beat_rate*100:>14.0f}% {avg_excess*100:>+14.1f}% {total_trades:>15}")
    
    wf_results.append({
        'threshold': thresh,
        'beat_rate': beat_rate,
        'avg_excess': avg_excess,
        'total_trades': total_trades
    })

wf_df = pd.DataFrame(wf_results)

In [ ]:
# Visualize
fig = go.Figure()

fig.add_trace(go.Bar(
    x=[f"NUPL > {t}" for t in wf_df['threshold']],
    y=wf_df['beat_rate'] * 100,
    marker_color=['green' if x > 0.62 else 'orange' if x > 0.54 else 'gray' for x in wf_df['beat_rate']],
    text=[f"{x:.0f}%" for x in wf_df['beat_rate']*100],
    textposition='outside'
))

fig.add_hline(y=54, line_dash='dash', line_color='orange', annotation_text='Baseline 54%')
fig.add_hline(y=62, line_dash='dash', line_color='green', annotation_text='MVRV exit 62%')

fig.update_layout(
    title='Walk-Forward Beat Rate: NUPL Exit vs MVRV Exit',
    yaxis_title='Beat Rate %',
    height=500
)
fig.show()

---
## 5. Compare NUPL vs MVRV Exit

In [ ]:
# MVRV baseline for comparison
def backtest_mvrv_exit(df, entries, mvrv_trigger=2.25, trailing_pct=0.20, stop_loss=0.20, max_hold_days=365):
    """MVRV exit for baseline comparison."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        trailing_active = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            
            if not trailing_active and current_mvrv >= mvrv_trigger:
                trailing_active = True
            
            if trailing_active:
                trail_stop = peak_price * (1 - trailing_pct)
                if current_price <= trail_stop:
                    exit_date = current_date
                    exit_price = trail_stop
                    exit_reason = 'mvrv_trail'
                    break
            
            if not trailing_active and stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({'pnl_pct': pnl, 'exit_reason': exit_reason})
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

# MVRV walk-forward
def walk_forward_mvrv(df, mvrv_trigger=2.25, trailing_pct=0.20):
    results = []
    close = df['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        sopr_sig = (test_df['sopr'] < 1) & (test_df['sopr_sth'] < 1)
        test_entries = sopr_sig & ~sopr_sig.shift(1).fillna(False)
        
        trades = backtest_mvrv_exit(test_df, test_entries, mvrv_trigger, trailing_pct)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({'beat_hold': strat_return > hold_return})
    
    return pd.DataFrame(results)['beat_hold'].mean()

print("\n\nDIRECT COMPARISON: NUPL vs MVRV EXIT")
print("="*60)

mvrv_beat = walk_forward_mvrv(df_test, 2.25, 0.20)
print(f"MVRV > 2.25 exit:  {mvrv_beat*100:.0f}% beat rate")

# Best NUPL
best_nupl = wf_df.loc[wf_df['beat_rate'].idxmax()]
print(f"NUPL > {best_nupl['threshold']} exit:  {best_nupl['beat_rate']*100:.0f}% beat rate")

---
## 6. Summary

In [ ]:
print("\n" + "="*80)
print("NUPL EXIT TRIGGER ANALYSIS SUMMARY")
print("="*80)

best_nupl = wf_df.loc[wf_df['beat_rate'].idxmax()]

print(f"\n📊 NUPL AS EXIT TRIGGER")
print(f"   Best threshold: NUPL > {best_nupl['threshold']}")
print(f"   Beat rate: {best_nupl['beat_rate']*100:.0f}%")
print(f"   Avg excess: {best_nupl['avg_excess']*100:+.1f}%")

print(f"\n📊 COMPARISON")
print(f"   MVRV > 2.25 exit:  62% beat rate (current best)")
print(f"   NUPL > {best_nupl['threshold']} exit:  {best_nupl['beat_rate']*100:.0f}% beat rate")

# Verdict
print(f"\n🎯 VERDICT:")
if best_nupl['beat_rate'] > 0.62:
    print(f"   ✅ NUPL exit BEATS MVRV exit!")
elif best_nupl['beat_rate'] >= 0.60:
    print(f"   ⚠️ NUPL exit similar to MVRV (both measure valuation)")
else:
    print(f"   ⚠️ NUPL exit doesn't improve on MVRV. Stick with MVRV > 2.25.")

print(f"\n💡 Note: NUPL and MVRV are highly correlated ({df['nupl'].corr(df['mvrv']):.2f})")
print(f"   They measure the same thing (unrealized profit) differently.")

print("\n" + "="*80)

In [ ]:
# Save results
import json

def to_native(obj):
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_native(v) for v in obj]
    elif hasattr(obj, 'item'):
        return obj.item()
    return obj

results = {
    'signal': 'nupl_exit',
    'walk_forward_results': to_native(wf_df.to_dict('records')),
    'best_nupl_exit': to_native(dict(best_nupl)),
    'baseline_mvrv': {'threshold': 2.25, 'beat_rate': 0.62},
    'nupl_mvrv_correlation': float(df['nupl'].corr(df['mvrv']))
}

with open('../data/nupl_exit_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved to ../data/nupl_exit_results.json")